# Function to automatic merge and fill missing values



## Create the function
You have to create a function which take 2 arguments :
- argument 1 : DataFrame main
- argument 2 : DataFrame opinion

Your function will return a new DataFrame which is the left merge of both DataFrames, and which has no missing values (so your function has to fill them), add new columns (flag, MONTH and DAY), and OPINION should be translated.

So you have to copy each useful code from the previous quests, and integrate it into a function. **This should achieve the same results as the previous quests.**

In [119]:
import pandas as pd

In [120]:
# Chargement des Datasets

link_main = "https://raw.githubusercontent.com/murpi/wilddata/master/quests/weather_main_2018.csv"
link_opinion = "https://raw.githubusercontent.com/murpi/wilddata/master/quests/weather_opinion_2018.csv"
df_main = pd.read_csv(link_main)
df_opinion = pd.read_csv(link_opinion)


In [121]:
# Affichage des premières lignes de df_main
df_main.head()


,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,PRESSURE_MAX_MB,CLOUDCOVER_AVG_PERCENT,HEATINDEX_MAX_C,DEWPOINT_MAX_C,WINDTEMP_MAX_C
0,2018-01-01,12,8,61,9,11,8,8.9,79,9.500,1018,41.750,12,8,7
1,2018-01-02,13,6,26,8,12,13,0.6,96,9.000,1020,87.875,13,12,6
2,2018-01-03,15,10,40,11,12,10,5.5,82,8.500,1017,91.500,15,13,7
3,2018-01-04,14,11,45,14,14,11,0.0,89,10.000,1011,90.125,14,12,10
4,2018-01-05,12,7,21,10,11,8,1.5,85,9.875,1005,62.375,12,10,7


In [122]:
# Affichage des premières lignes de df_opinion
df_opinion.head()

,date,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION
0,2018-01-01,113,0,3,5.1,bad
1,2018-03-12,119,0,2,8.8,bad
2,2018-03-09,116,0,3,10.2,bad
3,2018-10-07,122,0,1,5.6,bad
4,2018-06-18,119,0,1,12.9,bad


In [123]:
# Affichage du nombre de lignes et de colonnes de df_main
df_main.shape

(365, 15)

In [124]:
# Affichage des informations de df_main
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DATE                    365 non-null    str    
 1   MAX_TEMPERATURE_C       365 non-null    int64  
 2   MIN_TEMPERATURE_C       365 non-null    int64  
 3   WINDSPEED_MAX_KMH       365 non-null    int64  
 4   TEMPERATURE_MORNING_C   365 non-null    int64  
 5   TEMPERATURE_NOON_C      365 non-null    int64  
 6   TEMPERATURE_EVENING_C   365 non-null    int64  
 7   PRECIP_TOTAL_DAY_MM     365 non-null    float64
 8   HUMIDITY_MAX_PERCENT    365 non-null    int64  
 9   VISIBILITY_AVG_KM       365 non-null    float64
 10  PRESSURE_MAX_MB         365 non-null    int64  
 11  CLOUDCOVER_AVG_PERCENT  365 non-null    float64
 12  HEATINDEX_MAX_C         365 non-null    int64  
 13  DEWPOINT_MAX_C          365 non-null    int64  
 14  WINDTEMP_MAX_C          365 non-null    int64  
dtype

In [125]:
# Nombre de valeurs nulles de df_main
df_main.isna().sum()

DATE                      0
MAX_TEMPERATURE_C         0
MIN_TEMPERATURE_C         0
WINDSPEED_MAX_KMH         0
TEMPERATURE_MORNING_C     0
TEMPERATURE_NOON_C        0
TEMPERATURE_EVENING_C     0
PRECIP_TOTAL_DAY_MM       0
HUMIDITY_MAX_PERCENT      0
VISIBILITY_AVG_KM         0
PRESSURE_MAX_MB           0
CLOUDCOVER_AVG_PERCENT    0
HEATINDEX_MAX_C           0
DEWPOINT_MAX_C            0
WINDTEMP_MAX_C            0
dtype: int64

In [126]:
# Nombre de lignes et de colonnes de df_opinion
df_opinion.shape

(341, 6)

In [127]:
# Nombre de valeurs nulles de df_opinion
df_opinion.isna().sum()

date                    0
WEATHER_CODE_EVENING    0
TOTAL_SNOW_MM           0
UV_INDEX                0
SUNHOUR                 0
OPINION                 0
dtype: int64

In [128]:
# Création d'une fonction merge_and_fill qui prend en paramètre
# deux DataFrame pour les joindre avec un left join sur la colonne de date
from datetime import datetime


def merge_and_fill(DataFrameMain, DataFrameOpinion):  
  # MERGE BOTH DATAFRAMES
  # NewDataFrame = pd.merge(?)
  NewDataFrame = DataFrameMain.merge(DataFrameOpinion, how = "outer", left_on = 'DATE', right_on = 'date')
  NewDataFrame = NewDataFrame.drop(columns= "date")
  # FILL IN MISSING VALUES
  # WEATHER_CODE_EVENING => mode
  NewDataFrame = NewDataFrame.fillna({"WEATHER_CODE_EVENING": NewDataFrame.WEATHER_CODE_EVENING.mode()})

  # TOTAL_SNOW_MM => moyenne
  NewDataFrame = NewDataFrame.fillna({"TOTAL_SNOW_MM": NewDataFrame.TOTAL_SNOW_MM.mean()})

  # UV_INDEX => médiane
  NewDataFrame = NewDataFrame.fillna({"UV_INDEX": NewDataFrame.UV_INDEX.median()})

  # SUNHOUR => bfill
  NewDataFrame.SUNHOUR = NewDataFrame.SUNHOUR.bfill()

  # OPINION => remplacer les nuls par "Unknown"
  NewDataFrame["OPINION"] = NewDataFrame.OPINION.fillna("Unknown")

  # Ajout de la colonne FLAG pour OPINION nulles
  NewDataFrame["FLAG"] = NewDataFrame.OPINION.isna()

  # Conversion de la colonne DATE au format Datetime
  NewDataFrame["DATE"] = pd.to_datetime(NewDataFrame.DATE, format = "%Y-%m-%d")

  # Création de la colonne MONTH
  NewDataFrame["MONTH"] = NewDataFrame.DATE.map(lambda x: datetime.strftime(x, "%m"))

  # Création de la colonne DAY
  NewDataFrame["DAY"] = NewDataFrame.DATE.map(lambda x: datetime.strftime(x, "%d"))

  # Tradudction de la colonne OPINION en français (créer une fonction)
  def trad_op(opinion):
    if opinion == "bad":
      return "mauvais"
    elif opinion == "good":
          return "bon"
    elif opinion == "not good not bad":
              return "ni bon ni mauvais"
    elif opinion == "very bad":
              return "très mauvais"
    elif opinion == "very good":
              return "très bon"

  NewDataFrame.OPINION = NewDataFrame.OPINION.map(lambda x: trad_op(x))

  # Return new merged dataframe
  return NewDataFrame

In [129]:
df_2018 = merge_and_fill(df_main, df_opinion)

In [130]:
# Afficher le nombre de lignes et de colonnes de df_2018
print(df_2018.shape)
# Afficher les premières ignes de df_2018
display(df_2018.head())

(365, 23)


,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,DEWPOINT_MAX_C,WINDTEMP_MAX_C,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,FLAG,MONTH,DAY
0,2018-01-01,12,8,61,9,11,8,8.9,79,9.500,...,8,7,113.0,0.0,3.0,5.1,mauvais,False,01,01
1,2018-01-02,13,6,26,8,12,13,0.6,96,9.000,...,12,6,122.0,0.0,3.0,3.3,mauvais,False,01,02
2,2018-01-03,15,10,40,11,12,10,5.5,82,8.500,...,13,7,122.0,0.0,3.0,3.3,mauvais,False,01,03
3,2018-01-04,14,11,45,14,14,11,0.0,89,10.000,...,12,10,116.0,0.0,3.0,3.3,mauvais,False,01,04
4,2018-01-05,12,7,21,10,11,8,1.5,85,9.875,...,10,7,116.0,0.0,3.0,6.9,mauvais,False,01,05


In [131]:
# Afficher les informations de df_2018
df_2018.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   DATE                    365 non-null    datetime64[us]
 1   MAX_TEMPERATURE_C       365 non-null    int64         
 2   MIN_TEMPERATURE_C       365 non-null    int64         
 3   WINDSPEED_MAX_KMH       365 non-null    int64         
 4   TEMPERATURE_MORNING_C   365 non-null    int64         
 5   TEMPERATURE_NOON_C      365 non-null    int64         
 6   TEMPERATURE_EVENING_C   365 non-null    int64         
 7   PRECIP_TOTAL_DAY_MM     365 non-null    float64       
 8   HUMIDITY_MAX_PERCENT    365 non-null    int64         
 9   VISIBILITY_AVG_KM       365 non-null    float64       
 10  PRESSURE_MAX_MB         365 non-null    int64         
 11  CLOUDCOVER_AVG_PERCENT  365 non-null    float64       
 12  HEATINDEX_MAX_C         365 non-null    int64         
 13  D

## Execution with a loop
You will execute your function in a 9 rounds loop, once per year, and agregate this in a new big DataFrame with 24 columns and lots of rows.

In [132]:
# Chaque année entre 2010 et 2018 (inclus) a deux 2 sources : weather_main_YYYY.csv and weather_opinion_YYYY.csv
# Charger les deux datasets et joingez-les grâce à la fonction merge_and_fill
# Concaténez ce nouveau DataFrame au(x) précédent(s) pour n'obtenir qu'un long DataFrame df_weather

df_weather = pd.DataFrame()

# Créez la boucle :
for year in range(2010, 2019):
  main = pd.read_csv(f"https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather_main_{year}.csv")
  opinion = pd.read_csv(f"https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather_opinion_{year}.csv")
  
  df_weather = pd.concat([df_weather, merge_and_fill(main, opinion)], ignore_index = True)


In [133]:
# Affichage des premières lignes de df_weather
df_weather.head()

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,DEWPOINT_MAX_C,WINDTEMP_MAX_C,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,FLAG,MONTH,DAY
0,2010-01-01,5,2,19,2,5,3,0.0,86,10.000,...,3,-1,119.0,0.0,1.0,5.1,très mauvais,False,01,01
1,2010-01-02,5,-1,6,-1,4,1,0.0,78,10.000,...,0,2,116.0,0.0,1.0,8.7,très mauvais,False,01,02
2,2010-01-03,5,1,26,2,4,2,0.3,61,9.875,...,0,-3,119.0,0.0,1.0,6.9,très mauvais,False,01,03
3,2010-01-04,1,-4,25,-2,0,-3,0.0,75,10.000,...,0,-9,113.0,0.0,1.0,8.7,très mauvais,False,01,04
4,2010-01-05,2,-5,20,-5,1,-4,0.0,71,10.000,...,0,-10,113.0,0.0,1.0,8.7,très mauvais,False,01,05


In [134]:
df_weather.OPINION.unique()

<StringArray>
['très mauvais', nan, 'mauvais', 'ni bon ni mauvais', 'bon', 'très bon']
Length: 6, dtype: str

In [135]:
# Création d'une fonction clean qui :
# * crée une colonne FLAG indiquant une valeur manquante
# * convertit la colonne DATE au format datetime
# * traduit l'opinion en français

def clean(DataFrameYear):

  # ALIGN DATAFRAME SHAPE, DATATYPE
  # Colonne FLAG
  DataFrameYear["FLAG"] = DataFrameYear.OPINION.isna()

  # Colone DATE
  DataFrameYear["DATE"] = DataFrameYear.DATE.map(lambda x: datetime.strptime(x, "%Y-%m-%d"))
  
  # Colonne OPINION
  def trad_op(opinion):
      if opinion == "bad":
        return "mauvais"
      elif opinion == "good":
        return "bon"
      elif opinion == "not good not bad":
        return "ni bon ni mauvais"
      elif opinion == "very bad":
        return "très mauvais"
      elif opinion == "very good":
        return "très bon"
      else:
        return opinion
  
  DataFrameYear["OPINION"] = DataFrameYear.OPINION.map(lambda x: trad_op(x))
  
  # Return new cleaned dataframe
  return DataFrameYear

In [136]:
# On crée une liste de DataFrames pour les années 2019 à 2022 (inclus)

links = {}
for year in range(2019, 2023):
    # Collect both DataFrames each year
    year = str(year)
    links[year] = "https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather" + year + ".csv"

# Create dataframes from the links to clean
dfs = {}  # Dictionary to store dataframes
for year, link in links.items():
    dfs[year] = pd.read_csv(link)
    dfs[year] = clean(dfs[year])

dfs['2019']

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,WEATHER_CODE_MORNING,WEATHER_CODE_NOON,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,MONTH,DAY,FLAG
0,2019-01-01,9,4,10,4,7,8,0.2,94,9.000,...,116,143,176,0,1,5.1,très mauvais,1,1,False
1,2019-01-02,8,5,18,7,7,5,0.0,90,9.000,...,119,116,116,0,1,8.7,très mauvais,1,2,False
2,2019-01-03,6,0,18,0,4,3,0.0,88,10.000,...,116,116,116,0,1,8.7,très mauvais,1,3,False
3,2019-01-04,5,-1,15,-1,4,3,0.0,91,10.000,...,116,116,122,0,1,5.1,très mauvais,1,4,False
4,2019-01-05,6,-1,8,-1,4,3,0.0,91,8.000,...,143,116,116,0,1,8.7,très mauvais,1,5,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2019-12-27,13,10,13,10,12,11,0.0,96,7.375,...,119,143,116,0,1,8.7,ni bon ni mauvais,12,27,False
361,2019-12-28,11,5,12,7,8,8,0.0,93,7.625,...,143,116,116,0,1,6.9,très mauvais,12,28,False
362,2019-12-29,9,2,11,3,6,7,0.0,86,10.000,...,116,116,116,0,1,8.7,très mauvais,12,29,False
363,2019-12-30,12,4,7,5,9,9,0.0,61,10.000,...,113,113,113,0,1,8.7,très mauvais,12,30,False


In [137]:
# Concaténez df_weather avec df_2019 à 2022
for years in dfs:
    df_weather = pd.concat([df_weather, dfs[str(years)]], ignore_index = True)

In [138]:
# Affichez les premières lignes de df_weather
print(df_weather)

           DATE  MAX_TEMPERATURE_C  MIN_TEMPERATURE_C  WINDSPEED_MAX_KMH  \
0    2010-01-01                  5                  2                 19   
1    2010-01-02                  5                 -1                  6   
2    2010-01-03                  5                  1                 26   
3    2010-01-04                  1                 -4                 25   
4    2010-01-05                  2                 -5                 20   
...         ...                ...                ...                ...   
4743 2022-12-27                 10                  3                 19   
4744 2022-12-28                 12                  7                 35   
4745 2022-12-29                 12                  8                 18   
4746 2022-12-30                 15                  7                 47   
4747 2022-12-31                 17                 14                 32   

      TEMPERATURE_MORNING_C  TEMPERATURE_NOON_C  TEMPERATURE_EVENING_C  \
0            

# Draw a scatterplot
With Date on X-Axis and Max Temperature on Y-axis

The result must have 11 seasonality for the 11 summers/winters alternance.

Utilisez Seaborn ou Plotly Express selon votre préférence




In [140]:
import plotly.express as px

fig = px.scatter(df_weather, x='DATE', y='MAX_TEMPERATURE_C', title="Évolution de la Température Maximale")
fig.show()



Find appropriate dataviz to answer to this question :

Which year had the best weather? Are years very different or not ?


In [ ]:
df_weather.

In [146]:
fig = px.scatter(df_weather, x='DATE', y='PRECIP_TOTAL_DAY_MM', title="Pluie journalière")
fig.show()

In [144]:
"""
L'année 2022 semble avoir été une bonne année car il y a eu peu de pluie
"""

"\nL'année 2022 semble avoir été une bonne année car il y a eu peu de pluie\n"

Quelles années semblent avoir bénéficié des meilleures conditions météo ? Pour quelles raisons ?